In [23]:
# !pip install scholarly
# !pip install scrape-google-scholar-py

In [ ]:
import re
import requests
from bs4 import BeautifulSoup

import time
import requests
import pandas as pd
from collections import defaultdict

def get_paper_info_semantic(title, delay=2):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    params = {
        "query": title,
        "fields": "title,authors,year,venue,abstract,url,citationCount"
    }

    # Delay before making the request
    # print(f"Waiting for {delay} seconds before querying...")
    time.sleep(delay)

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print("Error:", response.status_code, response.text)
        return

    data = response.json()
    if data["total"] == 0 or not data["data"]:
        print("No paper found for:", title)
        return

    paper = data["data"][0]  # First matching result

    paper = data["data"][0]  # First matching result

    paper_dict = defaultdict()
    if paper.get("citationCount") > 15:
        paper_dict['title'] = paper.get("title")
        paper_dict['authors'] = ", ".join([author["name"] for author in paper.get("authors", [])])
        paper_dict['year'] = paper.get("year")
        paper_dict['venue'] = paper.get("venue")
        paper_dict['abstract'] = paper.get("abstract")
        paper_dict['citation_count'] = paper.get("citationCount")
        paper_dict['url'] = paper.get("url")
        return dict(paper_dict)

years = ["2023", "2024", "2025"]
venues = ['iclr', 'icml', 'neurips']
papers = list()
for year in years:
    for venue in venues:
        # Step 1: Get the webpage content
        url = f"https://{venue}.cc/virtual/{year}/papers.html?search=reinforcement+learning"
        response = requests.get(url)

        # Step 2: Parse the HTML content
        soup = BeautifulSoup(response.text, 'html.parser')

        # Pattern to match href like /virtual/2025/poster/XXXXX
        pattern = re.compile(rf"^/virtual/{year}/poster/\d+$")

        # Find all <li> tags with the desired <a> href
        matched_li_tags = soup.find_all('li', recursive=True)
        
        for li in matched_li_tags:
            a_tag = li.find('a', href=pattern)
            if a_tag and ("reinforcement learning" in a_tag.get_text(strip=True).lower()):
                papers.append(a_tag.get_text(strip=True))

In [46]:
# papers

In [ ]:
# Example usage
paper_dicts = []
for paper in papers[738:]:
    paper_dict = get_paper_info_semantic(paper, delay=35)
    if paper_dict is not None:
        paper_dicts.append(paper_dict)

pd.DataFrame(paper_dicts).to_excel('papers2.xlsx')


In [49]:
pd.DataFrame(paper_dicts).to_excel('papers.xlsx')

In [51]:
paper
papers.index(paper)
    

737

In [54]:
papers[738:]

['Iteratively Refined Behavior Regularization for Offline Reinforcement Learning',
 'Differentially Private Reinforcement Learning with Self-Play',
 'The Value of Reward Lookahead in Reinforcement Learning',
 'Diffusion-based Reinforcement Learning via Q-weighted Variational Policy Optimization',
 'Maximum Entropy Inverse Reinforcement Learning of Diffusion Models with Energy-Based Models',
 'Balancing Context Length and Mixing Times for Reinforcement Learning at Scale',
 'Kernel-Based Function Approximation for Average Reward Reinforcement Learning: An Optimist No-Regret Algorithm',
 'Contextual Bilevel Reinforcement Learning for Incentive Alignment',
 'Efficient Reinforcement Learning by Discovering Neural Pathways',
 'Hybrid Reinforcement Learning Breaks Sample Size Barriers In Linear MDPs',
 'Subwords as Skills: Tokenization for Sparse-Reward Reinforcement Learning',
 'When Your AIs Deceive You: Challenges of Partial Observability in Reinforcement Learning from Human Feedback',
 'C